In [1]:
import s3fs
import os
import json
import mimetypes
import pandas as pd
import geopandas as gpd
from dbfread import DBF
import lasio
import fsspec
from io import StringIO, BytesIO

from IPython.display import Image, display, IFrame
import matplotlib.pyplot as plt
%matplotlib inline
s3 = s3fs.S3FileSystem()
import boto3
s3_client = boto3.client('s3')

### Look_up file

In [2]:
s3_path = "s3://ayata-clients/Baytex_new/raw_data/PatchIQ/Lookup/ranger equipment lookup.csv"
bucket_name = s3_path.split('/')[2]  # Extract bucket name
file_key = "/".join(s3_path.split('/')[3:])  # Extract file key

response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
csv_data = response['Body'].read().decode('utf-8')  # Decode bytes to string

lookup = pd.read_csv(StringIO(csv_data))

lookup

,item_id,point_name,equipment,parent_equipment_name,equipment_type
0,10000,Tubing Pressure,BIG FIVE A1 H,BIG FIVE Well Pad,Gas Lift
1,10001,Casing Pressure,BIG FIVE A1 H,BIG FIVE Well Pad,Gas Lift
2,10002,Tubing Pressure,BIG FIVE B 2H,BIG FIVE Well Pad,Gas Lift
3,10003,Casing Pressure,BIG FIVE B 2H,BIG FIVE Well Pad,Gas Lift
4,10004,Tubing Pressure,BLOODSTONE C 3H,BLOODSTONE C 3H D 4H Well Pad,Gas Lift
...,...,...,...,...,...
28454,71106,PLC Battery Voltage,PECAN Facility,NaN,Facility
28455,71107,PLC Battery Voltage,MATOCHA 1H Facility,NaN,Facility
28456,71108,PLC Battery Voltage,COLLEEN CALYPSO Facility,NaN,Facility
28457,71109,PLC Battery Voltage,HAWKEYE 9H 10H 11H Facility,NaN,Facility


### Reading History Folder under PatchIQ

In [ ]:
s3_path = "s3://ayata-clients/Baytex_new/raw_data/PatchIQ/History/"
bucket_name = s3_path.split('/')[2] 
prefix = "/".join(s3_path.split('/')[3:])

response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

if 'Contents' in response:
    file_paths = [f"s3://{bucket_name}/{obj['Key']}" for obj in response['Contents']]
    for file in file_paths:
        print(file)
else:
    print("No files found in the specified S3 folder.")

### Clean Path of all History files

In [ ]:
file_paths
file_paths.pop(0)
file_paths
cleaned_paths = [path.replace("s3://ayata-clients/", "") for path in file_paths]
cleaned_paths

#### Mergeing of all the files in History folder of PatchIQ to a csv

In [ ]:
 # Initialize S3 client
s3_client = boto3.client('s3')

# Define S3 bucket and folder path
s3_folder = "s3://ayata-clients/Baytex_new/raw_data/PatchIQ/History/"
bucket_name = s3_folder.split('/')[2]
prefix = "/".join(s3_folder.split('/')[3:])  # Extract folder prefix

# List all files in the folder
response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

# Extract file paths (excluding the folder itself)
file_keys = [obj['Key'] for obj in response.get('Contents', []) if not obj['Key'].endswith('/')]

# Define column names
column_names = ["ID", "Timestamp", "Value"]

# Create an empty list to store DataFrames
dfs = []

# Read each CSV file from S3 and append to the list
for file_key in file_keys:
    response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
    csv_data = response['Body'].read().decode('utf-8') 
    df = pd.read_csv(StringIO(csv_data), header=None, names=column_names, low_memory=False)
    dfs.append(df)

# Concatenate all DataFrames into one
final_df = pd.concat(dfs, ignore_index=True)

# Display the combined DataFrame
# print(final_df.head())

# Optionally, save the final DataFrame to a CSV file
final_df.to_csv("PatchIQ_History.csv", index=False)

#### Converting all the Files of History Folder to csv

In [ ]:
import boto3
import pandas as pd
import os
import re
from io import StringIO

# Initialize S3 client
s3_client = boto3.client('s3')

# Define S3 bucket and folder path
s3_folder = "s3://ayata-clients/Baytex_new/raw_data/PatchIQ/History/"
bucket_name = s3_folder.split('/')[2]
prefix = "/".join(s3_folder.split('/')[3:])  # Extract folder prefix

# Local directory to save CSV files
output_dir = "Patch_IQ_History"
os.makedirs(output_dir, exist_ok=True)  # Create folder if not exists

# List all files in the folder
response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

# Extract file paths (excluding the folder itself)
file_keys = [obj['Key'] for obj in response.get('Contents', []) if not obj['Key'].endswith('/')]

# Define column names
column_names = ["ID", "Timestamp", "Value"]

# Process each file individually
for file_key in file_keys:
    response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
    csv_data = response['Body'].read().decode('utf-8')

    # Read CSV data into a DataFrame
    df = pd.read_csv(StringIO(csv_data), header=None, names=column_names, low_memory=False)

    # Extract the file name (without folder path)
    original_file_name = os.path.basename(file_key)

    # Modify the filename by replacing dots in the timestamp with underscores
    modified_file_name = re.sub(r'\.(\d{3})', r'_\1', original_file_name)

    # Ensure it ends with .csv
    if not modified_file_name.endswith(".csv"):
        modified_file_name += ".csv"

    # Save the file as CSV in the output folder
    output_file = os.path.join(output_dir, modified_file_name)
    df.to_csv(output_file, index=False)

    print(f"Saved: {output_file}")

#### Saving the csv files to S3

In [ ]:
import boto3
import os

s3_client = boto3.client('s3')

local_folder = "Patch_IQ_History"  # Replace with your folder path
bucket_name = "ayata-clients"
s3_folder = "Baytex_new/ayata_processed_data/Patch_IQ_History"  # S3 folder path

def upload_folder_to_s3(local_folder, bucket_name, s3_folder):
    for root, dirs, files in os.walk(local_folder):
        for file in files:
            local_path = os.path.join(root, file)
            s3_path = os.path.join(s3_folder, os.path.relpath(local_path, local_folder))

            # Replace backslashes with forward slashes for S3 compatibility
            s3_path = s3_path.replace("\\", "/")

            print(f"Uploading {local_path} to s3://{bucket_name}/{s3_path}")
            s3_client.upload_file(local_path, bucket_name, s3_path)

upload_folder_to_s3(local_folder, bucket_name, s3_folder)


#### Reading Iron_IQ file

In [ ]:
import pandas as pd
iron_iQ = pd.read_excel('Iron IQ Well Name to Accounting ID Table.xlsx', engine='openpyxl')
iron_iQ

#### Joining of look_up and Iron_IQ

In [ ]:
outer_join_01 = pd.merge(lookup, iron_iQ, left_on='equipment', right_on='Name', how='left')
outer_join_01

#### Data without any NaN Value

In [ ]:
no_nan_rows = outer_join_01[outer_join_01.notna().all(axis=1)]
no_nan_rows

#### Data Contain NaN Value

In [ ]:
nan_rows = outer_join_01[outer_join_01.isna().any(axis=1)]
nan_rows

#### Accounting Number as NaN

In [ ]:
name_nan_rows = outer_join_01[outer_join_01['Accounting Number'].isna()]
name_nan_rows

#### Adding new column as Well Name which is basically a copy of the equipment in the lookup file

In [ ]:
lookup['well_name'] = lookup['equipment']
lookup

#### Removing some strings from the new columns

In [ ]:
import re

unwanted_strings = [
    'Water Meter', 'Oil Meter', 'Oil Tank 1', 'Gas Lift Meter', 'Water Tank'
    'Production Meter', 'Reject Tank', 'Oil Tank 2', 'Oil Tank 3', 'Water Tank #1', 'Water Tank #2',
    'Water Tank #3', 'Water Tank #4', 'Water Tank #5', 'Water Tank #6', 'Oil(Reject) Tank #5'
    'Oil Tank 4', 'Oil Tank 5', 'Meter', 'HP Meter', 'Jet Pump', 'Gas Lift', 'Drip Tank', 'Lact'
    'Remote Device', 'Oil Tank', 'Water Tank 1', 'Water Tank 2', 'Water Tank 3'
    'Cooler Scrubber', 'Separator', 'Fuel Scrubber', 'Oil(Reject) Tank #5'
    'Flare Scrubber', 'Facility', 'Flash Gas', 'Tank Battery', 'Heater Separator',
    'Heater Treater', 'Flare', 'Production', 'Water Test Tank', 'Water Common Tank',
    'Oil Test Tank', 'Oil Common Tank', 'Seperator', 'Bulk', 'Tank 1', 'Tank 2',
    'Oil Common Tank', 'Water Test Tank', 'Common', 'Production', 'Injection',
    'Compressor Discharge', 'Compressor Fuel Gas', 'Sales Check', 'Sales', 'Robin',
    'Unknown', 'Unknown Tank 3', 'Unknown Tank 4', 'Unknown Tank 5', 'Unknown Tank 6',
    'Unknown Tank 7', 'Unknown Tank 8', 'Water Tank', 'Remove Device', 'Cooler Scrubber', 'Scrubber',
    'Fuel', 'Tank', 'Water Tank #1', 'Water Tank #2', '#1', '#2', '#3', '#4', '#6', '#5', 'Gas Lift'
]

pattern = '|'.join(map(re.escape, unwanted_strings))
lookup['well_name'] = lookup['well_name'].str.replace(pattern, '', regex=True).str.strip()

#### Left join with the well_name of look_up file and Name in Iron_Iq file

In [ ]:
outer_join_02 = pd.merge(lookup, iron_iQ, left_on='well_name', right_on='Name', how='left')
outer_join_02

In [ ]:
no_nan_rows = outer_join_02[outer_join_02.notna().all(axis=1)]
no_nan_rows

In [ ]:
import pandas as pd
joined = pd.read_csv('joined.csv')
joined